In [2]:
# Author: Arthur Prigent
# Email: arthur.prigent@univ-brest.fr

In [3]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import glob
import datetime as dt
import gsw
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as mticker

In [4]:
# read
dir_clim = '/data0/user/aprigent/ISAS/ISAS17_Mask.nc'
mask = xr.open_dataset(dir_clim)
zi = mask.depthCFD.data

# Vertical interpolation function


In [5]:
import numpy as np
from scipy.interpolate import Akima1DInterpolator

def interp_prof(x_old, y_old, x_new):
    yi = []   # interpolated profiles
    iyi = []  # indices of successfully interpolated profiles
    
    for i in range(y_old.shape[0]):
        y = y_old[i, :]
        x = x_old[i, :]
        
        # Remove NaNs
        valid = ~np.isnan(x) & ~np.isnan(y)
        x_valid = x[valid]
        y_valid = y[valid]
        
        if len(x_valid) < 3:
            continue  # not enough points to interpolate
        
        # Sort and remove duplicates
        sort_idx = np.argsort(x_valid)
        x_sorted = x_valid[sort_idx]
        y_sorted = y_valid[sort_idx]
        _, unique_idx = np.unique(x_sorted, return_index=True)
        x_unique = x_sorted[unique_idx]
        y_unique = y_sorted[unique_idx]
        
        if len(x_unique) < 2:
            continue  # still not enough unique points
        #print(x_unique.shape,y_unique.shape)
        # Interpolate
        spl = Akima1DInterpolator(x_unique, y_unique)
        prof = spl(x_new)
        
        yi.append(prof)
        iyi.append(i)
        
    return np.array(yi), np.array(iyi)

# Create the list of the Argo floats number for exclusion in CORA

In [6]:

argo_numbers = []

with open("/data0/user/aprigent/ARGO/Argo_Arctic_floats_03032026.txt") as f:
    for line in f:
        argo_numbers.append(int(line.strip().split("/")[-1]))

print(len(argo_numbers))


573


# Start looping on CORA CTD individual files

In [8]:
import numpy as np
import xarray as xr
import datetime as dt
import gsw
import os

no_data_files = []

for year in range(1980, 2015, 1):
    print('----------> current year = ', year)

    cora_path = '/data0/user/aprigent/CORA/' + str(year) + '/'
    list_file = glob.glob(cora_path + 'CO_DMQCGL01*_PR_CT*.nc')
    data_out  = '/data0/user/aprigent/CORA/processed/'

    # -------------------------------------------------------
    # ACCUMULATORS — collect all valid profiles for this year
    # -------------------------------------------------------
    all_lat_t,  all_lon_t,  all_time_t, all_temp, sum_t, sum_dep_t, nb_t, platform_t = [], [], [], [], [], [], [], []
    all_lat_s,  all_lon_s,  all_time_s, all_psal, sum_s, sum_dep_s, nb_s, platform_s = [], [], [], [], [], [], [], []

    for f in list_file:

        cora = xr.open_dataset(f, decode_times=False)

        if cora.N_LEVELS.shape[0] == 0:
            print(f, 'Has 0 vertical levels')
            continue
            
        platform = np.char.strip(cora.PLATFORM_NUMBER.values.astype(str))
        mask_itp = ~np.char.startswith(platform, 'ITP')
        mask_top = ~np.char.startswith(platform, 'TOP')
        # mask for floats not in exclusion list
        mask_exclude = np.array([
        (p.isdigit() and int(p) not in argo_numbers) or (not p.isdigit())
        for p in platform])
        
        excluded_count = (mask_exclude).sum()
        print(f, 'Excluded floats from list:', excluded_count)
        removed = platform[~mask_exclude]
        print("Removed platform numbers:", np.unique(removed))
        # combine all masks
        mask = mask_itp & mask_top & mask_exclude
        
        # print(f, 'Has', mask_itp.sum(), 'not ITP profiles')
        # print(f, 'Has', mask_top.sum(), 'not TOP profiles')
        # print(f, 'Has', mask_exclude.sum(), 'not excluded floats')
        print(f, 'Has', mask.sum(), 'remaining number of profiles')
        # remove ITP profiles
        cora = cora.isel(N_PROF=mask)
        platform_out_tmp = np.char.strip(cora.PLATFORM_NUMBER.values.astype(str))
        platform_out = np.char.add('CORA_', platform_out_tmp).astype('S30')
        if cora.N_PROF.shape[0]==0:
            print(f, 'Has 0 profiles after ITP removal')
            continue


        print('Processing:', f)

        # Time conversion (Julian days since 1950-01-01 to tordinal)
        juld_ord = np.array([
            dt.datetime(1950, 1, 1).toordinal() + t for t in cora.TIME.values
        ])

        lat = cora.LATITUDE.values
        lon = cora.LONGITUDE.values
        


        # Vertical coordinate selection (PRES preferred over DEPH)
        if 'PRES' in cora:
            pres = cora.PRES.where(cora.PRES_QC == 1).values
        else:
            pres = None

        if 'DEPH' in cora:
            deph = cora.DEPH.where(cora.DEPH_QC == 1).values
        else:
            deph = None

        if pres is None and deph is None:
            print('No vertical coordinates')
            continue

        if pres is not None and np.isfinite(pres).sum() > 0:
            depth = -gsw.z_from_p(pres.T, lat).T
            print('Using PRES with', np.isfinite(pres).sum(), 'levels')
        elif deph is not None and np.isfinite(deph).sum() > 0:
            depth = deph
            print('Using DEPTH with', np.isfinite(deph).sum(), 'levels')
        
        sum_dep = np.nansum(depth,axis=1)
        
        n_levels= np.ones((depth.shape[0]))*depth.shape[0]

        
        # Ensure 2D shape (N_PROF, N_LEVELS)
        if depth.ndim == 1:
            depth = depth[np.newaxis, :]
            

        # QC filtering
        if 'TEMP' in cora:
            temp = cora.TEMP.where(
                (cora.TEMP_QC == 1) &
                (cora.TIME_QC == 1) &
                (cora.POSITION_QC == 1) &
                ('PRES_QC' not in cora or cora.PRES_QC == 1) &
                ('PROFILE_QC' not in cora or cora.PROFILE_QC == 1)
            )
            sum_temp = np.nansum(temp,axis=1)
            
            
        else:
            temp = None

        if 'PSAL' in cora:
            psal = cora.PSAL.where(
                (cora.PSAL_QC == 1) &
                (cora.TIME_QC == 1) &
                (cora.POSITION_QC == 1) &
                ('PRES_QC' not in cora or cora.PRES_QC == 1) &
                ('PROFILE_QC' not in cora or cora.PROFILE_QC == 1)
            )
            sum_psal  = np.nansum(psal,axis=1)
        else:
            psal = None

        # Ensure 2D shape
        temp_vals = temp.values if temp is not None else None
        psal_vals = psal.values if psal is not None else None
        if temp_vals is not None and temp_vals.ndim == 1:
            temp_vals = temp_vals[np.newaxis, :]
        if psal_vals is not None and psal_vals.ndim == 1:
            psal_vals = psal_vals[np.newaxis, :]

        has_temp = temp_vals is not None and np.isfinite(temp_vals).sum() > 0
        has_psal = psal_vals is not None and np.isfinite(psal_vals).sum() > 0

        if not has_temp and not has_psal:
            print('skip', f)
            no_data_files.append(f)
            continue

        # Interpolation and accumulation — TEMP
        if has_temp:
            TEMP, itemp = interp_prof(depth, temp_vals, zi)
            if len(itemp) == 0:
                print('No valid TEMP profiles after interpolation')
            else:
                TEMP[TEMP < -2] = np.nan
                TEMP[TEMP > 30] = np.nan
                all_temp.append(TEMP)
                sum_t.append(sum_temp[itemp])
                sum_dep_t.append(sum_dep[itemp])
                nb_t.append(n_levels[itemp])
                all_time_t.append(juld_ord[itemp])
                platform_t.append(platform_out[itemp])
                all_lon_t.append(lon[itemp])
                all_lat_t.append(lat[itemp])

                print('TEMP profiles added:', len(itemp))

        # Interpolation and accumulation — PSAL
        if has_psal:
            PSAL, ipsal = interp_prof(depth, psal_vals, zi)
            if len(ipsal) == 0:
                print('No valid PSAL profiles after interpolation')
            else:
                PSAL[PSAL < 0]  = np.nan
                PSAL[PSAL > 40] = np.nan
                all_psal.append(PSAL)
                sum_s.append(sum_psal[ipsal])
                sum_dep_s.append(sum_dep[ipsal])
                nb_s.append(n_levels[ipsal])
                all_time_s.append(juld_ord[ipsal])
                platform_s.append(platform_out[ipsal])
                all_lon_s.append(lon[ipsal])
                all_lat_s.append(lat[ipsal])
                print('PSAL profiles added:', len(ipsal))

    # -------------------------------------------------------
    # SAVE ANNUAL TEMPERATURE FILE
    # -------------------------------------------------------
    if len(all_temp) > 0:
        TEMP_year = np.concatenate(all_temp,  axis=0)
        TIME_year = np.concatenate(all_time_t, axis=0)
        sum_T_year = np.concatenate(sum_t,  axis=0)
        sum_dep_T_year = np.concatenate(sum_dep_t, axis=0)
        np_dep_T_year = np.concatenate(nb_t, axis=0)
        LON_year  = np.concatenate(all_lon_t,  axis=0)
        LAT_year  = np.concatenate(all_lat_t,  axis=0)
        PLATFORM_t_year = np.concatenate(platform_t,  axis=0)
        

        ds = xr.Dataset(
            {
                'latitude':    (['prof'], LAT_year),
                'longitude':   (['prof'], LON_year),
                'time':        (['prof'], TIME_year),
                'temperature': (['prof', 'levels'], TEMP_year),
                'prof_descr':        (['prof'], PLATFORM_t_year),
                'sum_temperature': (['prof'], sum_T_year),
                'sum_levels': (['prof'], sum_dep_T_year),
                'nb_levels': (['prof'], np_dep_T_year),
                'depth':       (['levels'], zi)
            },
            coords={
                'prof':   np.arange(len(TIME_year)),
                'levels': zi
            }
        )
        ds.attrs['Comments']      = 'CORA temperature profiles (qc=1) interpolated on ISAS vertical grid using Akima interpolation'
        ds.attrs['title']         = 'CORA temperature profiles on ISAS vertical grid'
        ds.attrs['source']        = 'CORA'
        ds.attrs['history']       = 'Created with xarray'
        ds.attrs['year']          = str(year)

        ds['latitude'].attrs    = {'long_name': 'Latitude',    'units': 'degrees_north', 'standard_name': 'latitude'}
        ds['longitude'].attrs   = {'long_name': 'Longitude',   'units': 'degrees_east',  'standard_name': 'longitude'}
        ds['time'].attrs        = {'long_name':'Time', "units": "days since 0001-01-01 (Python datetime system)"}
        ds['temperature'].attrs = {'long_name': 'Temperature (T90)', 'units': 'degree_Celsius', 'standard_name': 'sea_water_temperature'}
        ds['depth'].attrs       = {'long_name': 'Depth', 'units': 'm', 'positive': 'down', 'standard_name': 'depth'}

        outfile = data_out + f'CORA_{year}_CT_ISAS_TEMP.nc'
        ds.to_netcdf(outfile)
        print(f'Saved {len(TIME_year)} TEMP profiles → {outfile}')
    else:
        print(f'No valid TEMP profiles for {year}')

    # -------------------------------------------------------
    # SAVE ANNUAL SALINITY FILE
    # -------------------------------------------------------
    if len(all_psal) > 0:
        PSAL_year = np.concatenate(all_psal,  axis=0)
        TIME_year = np.concatenate(all_time_s, axis=0)
        sum_S_year = np.concatenate(sum_s,  axis=0)
        sum_dep_S_year = np.concatenate(sum_dep_s, axis=0)
        np_dep_S_year = np.concatenate(nb_s, axis=0)
        LON_year  = np.concatenate(all_lon_s,  axis=0)
        LAT_year  = np.concatenate(all_lat_s,  axis=0)
        PLATFORM_s_year = np.concatenate(platform_s,  axis=0)
        

        ds = xr.Dataset(
            {
                'latitude':  (['prof'], LAT_year),
                'longitude': (['prof'], LON_year),
                'time':      (['prof'], TIME_year),
                'salinity':  (['prof', 'levels'], PSAL_year),
                'prof_descr':        (['prof'], PLATFORM_s_year),
                'sum_salinity': (['prof'], sum_S_year),
                'sum_levels': (['prof'], sum_dep_S_year),
                'nb_levels': (['prof'], np_dep_S_year),
                'depth':     (['levels'], zi)
            },
            coords={
                'prof':   np.arange(len(TIME_year)),
                'levels': zi
            }
        )
        ds.attrs['Comments']      = 'CORA salinity profiles (qc=1) interpolated on ISAS vertical levels using Akima interpolation'
        ds.attrs['title']         = 'CORA salinity profiles on ISAS vertical grid'
        ds.attrs['source']        = 'CORA'
        ds.attrs['history']       = 'Created with xarray'
        ds.attrs['year']          = str(year)

        ds['latitude'].attrs  = {'long_name': 'Latitude',  'units': 'degrees_north', 'standard_name': 'latitude'}
        ds['longitude'].attrs = {'long_name': 'Longitude', 'units': 'degrees_east',  'standard_name': 'longitude'}
        ds['time'].attrs      = {'long_name':'Time', "units": "days since 0001-01-01 (Python datetime system)"}
        ds['salinity'].attrs  = {'long_name': 'Salinity (S78 - PSS)', 'units': '1e-3', 'standard_name': 'sea_water_practical_salinity'}
        ds['depth'].attrs     = {'long_name': 'Depth', 'units': 'm', 'positive': 'down', 'standard_name': 'depth'}

        outfile = data_out + f'CORA_{year}_CT_ISAS_PSAL.nc'
        ds.to_netcdf(outfile)
        print(f'Saved {len(TIME_year)} PSAL profiles → {outfile}')
    else:
        print(f'No valid PSAL profiles for {year}')

----------> current year =  1980
/data0/user/aprigent/CORA/1980/CO_DMQCGL01_19800114_PR_CT.nc Excluded floats from list: 2
Removed platform numbers: []
/data0/user/aprigent/CORA/1980/CO_DMQCGL01_19800114_PR_CT.nc Has 2 remaining number of profiles
Processing: /data0/user/aprigent/CORA/1980/CO_DMQCGL01_19800114_PR_CT.nc
Using PRES with 100 levels
TEMP profiles added: 2
PSAL profiles added: 2
/data0/user/aprigent/CORA/1980/CO_DMQCGL01_19800422_PR_CT.nc Excluded floats from list: 1
Removed platform numbers: []
/data0/user/aprigent/CORA/1980/CO_DMQCGL01_19800422_PR_CT.nc Has 1 remaining number of profiles
Processing: /data0/user/aprigent/CORA/1980/CO_DMQCGL01_19800422_PR_CT.nc
Using DEPTH with 7 levels
skip /data0/user/aprigent/CORA/1980/CO_DMQCGL01_19800422_PR_CT.nc
/data0/user/aprigent/CORA/1980/CO_DMQCGL01_19800801_PR_CT.nc Excluded floats from list: 24
Removed platform numbers: []
/data0/user/aprigent/CORA/1980/CO_DMQCGL01_19800801_PR_CT.nc Has 24 remaining number of profiles
Processin

In [30]:
import glob
import xarray as xr
import numpy as np
def concat_netcdf_psal(file_list):
    ds = xr.open_dataset(file_list[0],decode_times=False)
    param = ds.salinity.data

    lon = ds.longitude.data
    lat = ds.latitude.data
    time = ds.time.data
    depth = ds.depth.data
    
    for f in file_list[1:]:
        ds = xr.open_dataset(f,decode_times=False)
        param = np.concatenate((param, ds.salinity.data),axis=0)
        lon = np.concatenate((lon, ds.longitude.data),axis=0)
        lat = np.concatenate((lat, ds.latitude.data),axis=0)
        time = np.concatenate((time, ds.time.data),axis=0)
        
    return param,lon,lat,time,depth

    
def concat_netcdf_temp(file_list):
    ds = xr.open_dataset(file_list[0],decode_times=False)
    param = ds.temperature.data
    lon = ds.longitude.data
    lat = ds.latitude.data
    time = ds.time.data
    depth = ds.depth.data
    
    for f in file_list[1:]:
        ds = xr.open_dataset(f,decode_times=False)
        param = np.concatenate((param, ds.temperature.data),axis=0)
        lon = np.concatenate((lon, ds.longitude.data),axis=0)
        lat = np.concatenate((lat, ds.latitude.data),axis=0)
        time = np.concatenate((time, ds.time.data),axis=0)
        
    return param,lon,lat,time,depth

itp_path = '/data0/user/aprigent/CORA/processed/'
itp_list_psal = glob.glob(itp_path + '*_CT_ISAS_PSAL.nc')
itp_list_temp = glob.glob(itp_path + '*_CT_ISAS_TEMP.nc')
psal_itp,lon_itp_psal,lat_itp_psal,time_itp_psal, dep_psal = concat_netcdf_psal(itp_list_psal)
temp_itp,lon_itp_temp,lat_itp_temp,time_itp_temp, dep_temp = concat_netcdf_temp(itp_list_temp)